In [2]:
import pandas as pd
import ast
import numpy as np
import os
from itertools import islice
from tqdm import tqdm
import json
from transformers import AutoTokenizer, AutoModel


## Entity normalization tests

### Load embeddings

In [3]:
directory_path = "./data/umls/embeddings"
batch_name_prefix = "UMLS_emb"

# List all files in the directory that match the pattern
files = [f for f in os.listdir(directory_path) if f.startswith(f'{batch_name_prefix}_batch_') and f.endswith('.npy')]
print(files)
# Sort files to maintain the order, especially important if the batch index is used in processing
files.sort()

# Initialize an empty list to hold the data from each file
all_data = []

# Load each file and append the data to the list
for file in files:
    file_path = os.path.join(directory_path, file)
    data = np.load(file_path)
    all_data.append(data)

all_reps_emb_full = np.concatenate(all_data, axis=0)

['UMLS_emb_batch_3.npy', 'UMLS_emb_batch_2.npy', 'UMLS_emb_batch_0.npy', 'UMLS_emb_batch_1.npy', 'UMLS_emb_batch_4.npy']


In [4]:
all_reps_emb_full.shape

(474316, 768)

In [5]:
with open("data/umls/umls_term_id_pairs.json", "r") as f:
    term_id_pairs = json.load(f)

In [6]:
len(term_id_pairs)

474316

# Test mappings

In [7]:
from neural_based_nen import map_query_to_terminology

In [8]:
tokenizer = AutoTokenizer.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")
model = AutoModel.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")

In [9]:
len(all_reps_emb_full), len(term_id_pairs)

(474316, 474316)

In [11]:
# Example usage
dist_threshold = 15
embeddings_to_use = all_reps_emb_full
corresponding_term_id = term_id_pairs
    
query = "interferon beta"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: C0015980
Predicted label: beta Interferon
Distance: 3.9498
Canonical form: beta Interferon
Nearest 3:  [['beta Interferon', 'C0015980'], ['Interferon beta (recombinant)', 'C0751599'], ['beta 1 Interferon', 'C0005148'], ['Interferon beta-1b', 'C0244713'], ['beta 1a, Interferon', 'C0254119']]


In [12]:
query = "inf beta 1"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: C0005148
Predicted label: beta 1 Interferon
Distance: 6.9799
Canonical form: beta 1 Interferon
Nearest 3:  [['beta 1 Interferon', 'C0005148'], ['beta 1a, Interferon', 'C0254119'], ['beta Interferon', 'C0015980'], ['Interferon beta (recombinant)', 'C0751599'], ['Interferon beta-1b', 'C0244713']]


In [13]:
query = "inf-beta"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: C0015980
Predicted label: beta Interferon
Distance: 6.9821
Canonical form: beta Interferon
Nearest 3:  [['beta Interferon', 'C0015980'], ['Interferon beta (recombinant)', 'C0751599'], ['beta 1 Interferon', 'C0005148'], ['Interferon beta-1b', 'C0244713'], ['beta 1a, Interferon', 'C0254119']]


In [14]:

query = "isorhynchophylline"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, None, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: C0245133
Predicted label: isorhynchophylline
Distance: 0.0
Canonical form: isorhynchophylline
Nearest 3:  [['isorhynchophylline', 'C0245133'], ['rhynchophylline', 'C0073214'], ['isomitraphylline', 'C1700154'], ['heterophylline', 'C1723057'], ['mitraphylline', 'C1701760']]
